In [14]:
# -*- coding: utf-8 -*-
from turtle import title
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tempfile
import os
from ansys.aedt.core import generate_unique_name
from ansys.aedt.core import Desktop, Hfss, constants

In [15]:
def SingleFrequency(
    freq,
    maxPasses,
    maxDeltaS= 0.02,
    name="MySetup"
):
    hfss.create_open_region(frequency=freq)
    setup.props["Frequency"] = freq
    setup.props["MaximumPasses"] = maxPasses
    setup.props["MaxDeltaS"] =maxDeltaS
    setup.props["UseIterativeSolver"] = True
    setup.props["SaveRadFieldsOnly"] = True
    setup.props["RadiationSetup"] = "3D"
    setup.props["ComputeFarFields"] = True

In [16]:
def LinearCountSweep(
    fStart,
    fStop,
    freqPonts=401,
    units="GHz",
    saveFields=True,
    saveRadFields=True
):
    sweep = hfss.create_linear_count_sweep(
        setup=setup.name,
        units=units,
        start_frequency=fStart,
        stop_frequency=fStop,
        num_of_freq_points = freqPonts,
        name= "sweep",
        sweep_type="Interpolating",
        save_fields=saveFields,
        save_rad_fields=saveRadFields
    )

In [17]:

def SaveAnalyseSetup(
    nameFolder= "DipoloAntenna.aedt",
    setup="MySetup"
):
    hfss.save_project(os.path.join(temp_folder, nameFolder))
    hfss.analyze_setup(setup)

In [18]:
tmpfold = tempfile.gettempdir()

temp_folder = os.path.join(
    tmpfold,
    generate_unique_name("Antenna")
)

if not os.path.exists(temp_folder):
    os.mkdir(temp_folder)

non_graphical = os.getenv(
    "PYAEDT_NON_GRAPHICAL",
    "False"
).lower() in ("true", "1", "t")

d = Desktop(
    "2025.1",
    non_graphical=non_graphical,
    new_desktop=True
)

L = 17.55
dR = 1.65
gL = 1
xSize = 17
ySize = 43
D = 3   
g = 0.434
L2 = 7.2

hfss = Hfss(solution_type="Modal")
hfss.modeler.model_units = "mm"

p = hfss.modeler.primitives

PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.17.1.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file C:\Users\LUIS~1.COS\AppData\Local\Temp\pyaedt_luis.costa_c4ddd419-1a6f-4330-8a3f-0de127474ff4.log is enabled.
PyAEDT INFO: Log on AEDT is enabled.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Launching PyAEDT with gRPC plugin.
PyAEDT INFO: New AEDT session is starting on gRPC port 52411.
PyAEDT INFO: Electronics Desktop started on gRPC port: 52411 after 16.931695461273193 seconds.
PyAEDT INFO: AEDT installation Path C:\Program Files\ANSYS Inc\v251\AnsysEM
PyAEDT INFO: Ansoft.ElectronicsDesktop.2025.1 version started with process ID 9708.
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT 

In [19]:
box = p.create_box(
    [-xSize/2, -ySize/2, 0],
    [xSize, ySize, dR],
    name="box",
    material="FR4_epoxy"
)

haste_1_horizontal = p.create_rectangle(
    constants.AXIS.Z,
    [L2-(xSize/2), -D/2, dR],
    (D, L),
    name="haste_1_h",
)

haste_1_vertical = p.create_rectangle(
    constants.AXIS.Z,
    [-xSize/2, -D/2, dR],
    (L2, D),
    name="haste_1_v",
)

haste_1 = p.unite([haste_1_horizontal, haste_1_vertical])

hfss.assign_perfect_e(
    haste_1_horizontal.name
)


haste_2_horizontal = p.create_rectangle(
    constants.AXIS.Z,
    [L2-(xSize/2), -L+D/2, 0],
    (D, L),
    name="haste_2_h",
)

haste_2_vertical = p.create_rectangle(
    constants.AXIS.Z,
    [-xSize/2, -D/2, 0],
    (L2, D),
    name="haste_2_v",
)

haste_2 = p.unite([haste_2_horizontal, haste_2_vertical])

port = p.create_rectangle(
    constants.AXIS.X,
    [-xSize/2, -D/2, 0],
    (D, dR),
    name="port",
)



PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Union of 2 objects has been executed.
PyAEDT INFO: Boundary Perfect E PerfectE_haste_1_h has been created.
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Union of 2 objects has been executed.


In [20]:
hfss.assign_perfect_e(
    haste_2_horizontal.name
)

hfss.lumped_port(
    assignment=port.name,                 
    reference=port,
    integration_line=hfss.AxisDir.ZNeg,    
    impedance=50,
    name=generate_unique_name("port1"),
    renormalize=False,
    terminals_rename=False
)

PyAEDT INFO: Boundary Perfect E PerfectE_haste_2_h has been created.
PyAEDT INFO: Boundary Lumped Port port1_X47OIK has been created.


In [21]:
my_plot =hfss.plot(show=False, plot_air_objects=False)
my_plot.show_axes = False
my_plot.show_grid = False
my_plot.isometric_view = True
my_plot.plot(
    os.path.join(hfss.working_directory, "DipoloAntenna.jpg")    
)

setup = hfss.create_setup("MySetup")

PyAEDT INFO: Parsing C:/Users/luis.costa/Documents/Ansoft/Project16.aedt.
PyAEDT INFO: PostProcessor class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: PostProcessor class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Post class has been initialized! Elapsed time: 0m 0sec


Widget(value='<iframe src="http://localhost:52445/index.html?ui=P_0x15183a6c580_0&reconnect=auto" class="pyvis…

In [22]:
SingleFrequency("5GHz", 30, 0.02) 

PyAEDT INFO: Open Region correctly created.
PyAEDT INFO: Project Project16 Saved correctly


In [23]:
LinearCountSweep(1.0,5.0)

PyAEDT INFO: Parsing C:/Users/luis.costa/Documents/Ansoft/Project16.aedt.
PyAEDT INFO: File C:/Users/luis.costa/Documents/Ansoft/Project16.aedt correctly loaded. Elapsed time: 0m 0sec
PyAEDT INFO: aedt file load time 0.12525463104248047
PyAEDT INFO: Linear count sweep sweep has been correctly created.


In [24]:
SaveAnalyseSetup("dipoloAntenna.aedt")

PyAEDT INFO: Project dipoloAntenna Saved correctly
PyAEDT INFO: Key Desktop/ActiveDSOConfigurations/HFSS correctly changed.
PyAEDT INFO: Solving design setup MySetup
PyAEDT INFO: Key Desktop/ActiveDSOConfigurations/HFSS correctly changed.
PyAEDT INFO: Design setup MySetup solved correctly in 0.0h 0.0m 43.0s


In [25]:
d.release_desktop()

PyAEDT INFO: Desktop has been released and closed.


True